# Dataset Augmentation with Synthetic Waste Records

This notebook expands the original food waste dataset by generating realistic synthetic records. The goal is to create a larger dataset that preserves the statistical properties of the real data while extending it to a longer time period (January 2015 to December 2024). The synthetic data includes:
- Continuous 30-minute time-series coverage (every bin has a value)
- Realistic waste values calibrated to the original distribution (log-normal parameters derived directly from the real data)
- Foot traffic patterns that vary by day of week, hour, and special events
- Event indicators to mark days with higher waste (e.g., campus events)
- Controlled noise and occasional outliers for organic variability
- Consistent 30-minute temporal granularity with no zero-padding gaps

The final output combines original and synthetic records into a single CSV file, ready for time-series analysis and forecasting.

## 1. Setup and Data Loading
We import required libraries and load the original cleaned dataset. A random seed is set to ensure reproducible results.

In [71]:
import os
import pandas as pd
import numpy as np
from datetime import timedelta, datetime

In [72]:
np.random.seed(42)

In [73]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Change working directory to project folder
try:
    os.chdir("/content/drive/MyDrive/UAB/FDS/campus-waste-intelligence")
    print("Directory changed")
except OSError:
    print("Error: Can't change the Current Working Directory")

Mounted at /content/drive
Directory changed


In [74]:
df_original = pd.read_csv('data/food_waste_cleaned.csv')
print(f"Original data shape: {df_original.shape}")

Original data shape: (2600, 23)


## 2. Add Realistic Times to Original Records
The original dataset only contains dates. We assign an hour and minute based on the recorded meal type to create timestamps that reflect typical meal times. This step makes the original data compatible with time-series analysis.

In [75]:
# Define meal time windows (start hour, end hour)
meal_time_windows = {
    "Breakfast": (6, 9),
    "Lunch": (11, 14),
    "Dinner": (17, 20)
}

def add_realistic_times(df):
    """
    Convert date column to datetime and add a realistic time within the meal window.
    Minutes are snapped to nearest 30-min bucket for time-series alignment.
    """
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'])

    def assign_time(row):
        start, end = meal_time_windows[row['Meal']]
        hour = np.random.randint(start, end)
        # Snap to 30-min bucket
        minute_bucket = np.random.choice([0, 30])
        return row['Date'] + timedelta(hours=int(hour), minutes=int(minute_bucket))

    df['Date'] = df.apply(assign_time, axis=1)
    return df

df_original = add_realistic_times(df_original)
print("First few timestamps after adding times:")
print(df_original['Date'].head())

First few timestamps after adding times:
0   2025-06-11 08:30:00
1   2025-06-11 11:00:00
2   2025-06-11 19:30:00
3   2025-06-11 11:00:00
4   2025-06-11 19:30:00
Name: Date, dtype: datetime64[ns]


## 3. Analyse Original Data
We examine the distributions of categorical and numerical variables from the original data. These statistics will guide the synthetic data generation to ensure it mimics real patterns.

**Critical**: The log-normal parameters for waste weight are computed directly from the raw data and used without additional scaling multipliers that would shrink the values.

In [76]:
# Categorical distributions
meal_dist = df_original['Meal'].value_counts(normalize=True)
section_dist = df_original['Canteen_Section'].value_counts(normalize=True)
category_dist = df_original['Food_Category'].value_counts(normalize=True)

# Compute conditional distributions (e.g., Food_Category given Meal)
category_given_meal = df_original.groupby('Meal')['Food_Category'].value_counts(normalize=True).unstack(fill_value=0)

print("Meal distribution:\n", meal_dist)
print("\nCanteen Section distribution:\n", section_dist)
print("\nFood Category distribution:\n", category_dist)
print("\nFood Category given Meal:\n", category_given_meal)

Meal distribution:
 Meal
Breakfast    0.334231
Lunch        0.333462
Dinner       0.332308
Name: proportion, dtype: float64

Canteen Section distribution:
 Canteen_Section
B    0.251538
A    0.250769
D    0.249231
C    0.248462
Name: proportion, dtype: float64

Food Category distribution:
 Food_Category
Rice          0.253462
Meat          0.251154
Vegetables    0.250385
Soup          0.245000
Name: proportion, dtype: float64

Food Category given Meal:
 Food_Category      Meat      Rice      Soup  Vegetables
Meal                                                   
Breakfast      0.245109  0.249712  0.255466    0.249712
Dinner         0.255787  0.261574  0.237269    0.245370
Lunch          0.252595  0.249135  0.242215    0.256055


In [77]:
# Unit price per food category (fixed)
price_map = {'Meat': 8.0, 'Vegetables': 3.0, 'Soup': 1.5, 'Rice': 2.0}

# -----------------------------------------------------------------
# Waste weight distribution per category: fit log-normal on the
# RAW Waste_Weight_kg values from the original data.
# We store (mean_log, std_log) which are the parameters of the
# underlying normal distribution after log-transform.
# These are used directly in generation – NO additional scaling.
# -----------------------------------------------------------------
df_original['Log_Waste'] = np.log(df_original['Waste_Weight_kg'])

category_params = {}
for cat in df_original['Food_Category'].unique():
    log_vals = df_original[df_original['Food_Category'] == cat]['Log_Waste']
    mu = log_vals.mean()
    sigma = log_vals.std()
    category_params[cat] = {'mean_log': mu, 'std_log': sigma}
    # Report both log-space params and the implied median in kg
    print(f"{cat}: log_mean={mu:.4f}, log_std={sigma:.4f}  => implied median={np.exp(mu):.2f} kg")

print(f"\nOriginal Waste_Weight_kg stats:")
print(df_original['Waste_Weight_kg'].describe())

Vegetables: log_mean=0.7206, log_std=0.8024  => implied median=2.06 kg
Soup: log_mean=0.7561, log_std=0.7875  => implied median=2.13 kg
Meat: log_mean=0.6427, log_std=0.8514  => implied median=1.90 kg
Rice: log_mean=0.7299, log_std=0.7933  => implied median=2.07 kg

Original Waste_Weight_kg stats:
count    2600.000000
mean        2.585988
std         1.408281
min         0.100000
25%         1.390000
50%         2.600000
75%         3.800000
max         5.000000
Name: Waste_Weight_kg, dtype: float64


In [78]:
# Compute baseline event counts per meal and day-of-week
df_original['DayOfWeek'] = df_original['Date'].dt.dayofweek
events_per_meal_day = df_original.groupby(['Date', 'Meal']).size().reset_index(name='count')
events_per_meal_day['DayOfWeek'] = events_per_meal_day['Date'].dt.dayofweek
baseline_events = events_per_meal_day.groupby(['Meal', 'DayOfWeek'])['count'].mean().round().astype(int)
# Guarantee at least 1 event per meal/day combination
baseline_events = baseline_events.clip(lower=1)
print("Baseline events per meal per day (by day of week):\n", baseline_events)

Baseline events per meal per day (by day of week):
 Meal       DayOfWeek
Breakfast  0            3
           1            3
           2            3
           3            3
           4            3
           5            3
           6            3
Dinner     0            3
           1            3
           2            3
           3            3
           4            3
           5            3
           6            3
Lunch      0            3
           1            3
           2            2
           3            3
           4            3
           5            3
           6            3
Name: count, dtype: int64


## 4. Define Global Thresholds
We calculate outlier and high-waste thresholds from the original data. These thresholds will be applied to both original and synthetic records to ensure consistent flagging.

In [79]:
def compute_thresholds(df):
    """
    Compute IQR-based outlier bounds and 90th percentile thresholds for waste and cost.
    """
    q1_waste = df['Waste_Weight_kg'].quantile(0.25)
    q3_waste = df['Waste_Weight_kg'].quantile(0.75)
    iqr_waste = q3_waste - q1_waste
    waste_lower = q1_waste - 1.5 * iqr_waste
    waste_upper = q3_waste + 1.5 * iqr_waste

    q1_cost = df['Cost_Loss'].quantile(0.25)
    q3_cost = df['Cost_Loss'].quantile(0.75)
    iqr_cost = q3_cost - q1_cost
    cost_lower = q1_cost - 1.5 * iqr_cost
    cost_upper = q3_cost + 1.5 * iqr_cost

    high_waste_thresh = df['Waste_Weight_kg'].quantile(0.90)
    high_cost_thresh = df['Cost_Loss'].quantile(0.90)

    return {
        'waste_lower': waste_lower,
        'waste_upper': waste_upper,
        'cost_lower': cost_lower,
        'cost_upper': cost_upper,
        'high_waste': high_waste_thresh,
        'high_cost': high_cost_thresh
    }

thresholds = compute_thresholds(df_original)
print(f"Waste outlier bounds: [{thresholds['waste_lower']:.2f}, {thresholds['waste_upper']:.2f}]")
print(f"Cost outlier bounds: [{thresholds['cost_lower']:.2f}, {thresholds['cost_upper']:.2f}]")
print(f"High waste threshold (90th percentile): {thresholds['high_waste']:.2f}")
print(f"High cost threshold (90th percentile): {thresholds['high_cost']:.2f}")

Waste outlier bounds: [-2.23, 7.42]
Cost outlier bounds: [-7.69, 22.13]
High waste threshold (90th percentile): 4.52
High cost threshold (90th percentile): 23.77


## 5. Define Helper Functions for Synthetic Generation

### Key design decisions for scale-correctness:
1. `generate_waste_weight` draws directly from the fitted log-normal and applies **only** a day-level multiplier (weekend/event). Meal and section factors are **removed** because they were compounding the log-normal scale in the original code, causing ~10x shrinkage versus the real data.
2. `get_foot_traffic_multiplier` returns a value near 1.0 for normal days so it does not distort waste magnitudes.
3. The 30-min continuous bins are generated in Section 7 by aggregating event-level records into fixed windows and filling gaps with small noise drawn from the tail of the observed low-waste distribution.

In [80]:
# Base foot traffic per section (rough estimates)
base_traffic_per_section = {'A': 100, 'B': 120, 'C': 80, 'D': 150}

def get_hour_multiplier(hour):
    """Return a multiplier for foot traffic based on hour of day."""
    if 6 <= hour <= 9:      # breakfast
        return 0.8
    elif 11 <= hour <= 14:  # lunch
        return 1.2
    elif 17 <= hour <= 20:  # dinner
        return 1.0
    else:
        return 0.1          # very low traffic outside meal hours

def get_foot_traffic_multiplier(date, event_days):
    """
    Return a multiplier that scales expected foot traffic.
    - Weekends: 0.7x
    - Event days: 1.8x
    - Normal weekday: 1.0x
    NOTE: This multiplier is used for foot_traffic calculation and
    event flagging only. It is NOT applied to waste weight generation
    to avoid distorting the calibrated log-normal distribution.
    """
    weekday = date.weekday()
    multiplier = 0.7 if weekday >= 5 else 1.0
    if date in event_days:
        multiplier *= 1.8
    return multiplier

In [81]:
def generate_event_count(meal, day_of_week, foot_traffic_multiplier, baseline_events):
    """
    Sample the number of waste events for a given meal on a specific day.
    Uses Poisson distribution with lambda = baseline * foot_traffic_multiplier.
    """
    base = baseline_events.loc[(meal, day_of_week)]
    lam = max(base * foot_traffic_multiplier, 0.5)  # floor lambda to avoid all-zero days
    return np.random.poisson(lam)

In [82]:
def generate_event_timestamp(base_date, meal):
    """
    Create a random timestamp within the meal window.
    The minute is snapped to the nearest 30-minute bucket (0 or 30)
    to align with the continuous 30-min grid.
    """
    start_hour, end_hour = meal_time_windows[meal]
    hour = int(np.random.randint(start_hour, end_hour))
    minute_bucket = int(np.random.choice([0, 30]))
    return base_date + timedelta(hours=hour, minutes=minute_bucket)

In [83]:
def generate_waste_weight(category, is_event_day=False):
    """
    Generate a waste weight for a single event, calibrated to the original data.

    Strategy:
    - Draw directly from the fitted log-normal (mean_log, std_log) of the category.
    - Apply a small event-day multiplier (1.2-1.5x) ONLY on event days.
    - Outliers are handled naturally by the log-normal tail; no artificial
      multiplier injection (which was causing max=79 vs original max=5).
    - Values are capped at the original data max (5 kg) to stay in-distribution.
    """
    params = category_params[category]
    log_waste = np.random.normal(params["mean_log"], params["std_log"])
    waste = np.exp(log_waste)

    # Event-day uplift
    if is_event_day:
        waste *= np.random.uniform(1.2, 1.5)

    # Cap at original data max to avoid out-of-distribution extremes
    waste = min(waste, 5.0)

    return max(waste, 0.01)  # ensure positive


In [84]:
def generate_derived_features(timestamp, waste_weight, unit_price, foot_traffic):
    """
    Compute all date-time derived columns and flag columns based on thresholds.
    """
    cost_loss = waste_weight * unit_price
    waste_per_price = waste_weight / unit_price if unit_price > 0 else 0
    log_waste = np.log(waste_weight)

    year          = timestamp.year
    month         = timestamp.month
    day           = timestamp.day
    hour          = timestamp.hour
    minute        = timestamp.minute
    weekday_num   = timestamp.weekday()
    weekday_name  = timestamp.strftime('%A')
    week          = timestamp.isocalendar()[1]
    day_of_year   = timestamp.timetuple().tm_yday
    quarter       = (month - 1) // 3 + 1
    is_weekend    = 1 if weekday_num >= 5 else 0
    month_name    = timestamp.strftime('%B')
    weekday_type  = 'Weekend' if is_weekend else 'Weekday'

    return {
        'Year': year, 'Month': month, 'Day': day, 'Hour': hour, 'Minute': minute,
        'Weekday': weekday_name, 'Week': week, 'DayOfYear': day_of_year, 'Quarter': quarter,
        'IsWeekend': is_weekend, 'Month_Name': month_name, 'Weekday_Type': weekday_type,
        'Cost_Loss': round(cost_loss, 2),
        'Waste_per_Price': round(waste_per_price, 6),
        'Log_Waste': round(log_waste, 6),
        'Foot_Traffic': round(foot_traffic, 2)
    }

In [85]:
def apply_flag_thresholds(df, thresholds):
    """
    Add boolean flag columns to the dataframe based on global thresholds.
    """
    df['Is_Waste_Outlier'] = ((df['Waste_Weight_kg'] < thresholds['waste_lower']) |
                              (df['Waste_Weight_kg'] > thresholds['waste_upper'])).astype(int)
    df['Is_Cost_Outlier']  = ((df['Cost_Loss'] < thresholds['cost_lower']) |
                              (df['Cost_Loss'] > thresholds['cost_upper'])).astype(int)
    df['Is_High_Waste']    = (df['Waste_Weight_kg'] > thresholds['high_waste']).astype(int)
    df['Is_High_Cost']     = (df['Cost_Loss'] > thresholds['high_cost']).astype(int)
    return df

## 6. Define Event Days
We randomly select some days within the target period to be event days. Event days simulate special occasions (e.g., campus festivals, sports events) that cause higher waste generation.

In [86]:
def create_event_days(start_date, end_date, n_events=100, seed=42):
    """
    Randomly select n_events unique days between start_date and end_date.
    Returns a set of Timestamp dates.
    """
    np.random.seed(seed)
    date_range = pd.date_range(start_date, end_date, freq='D')
    event_dates = np.random.choice(date_range, size=n_events, replace=False)
    # Normalise to date-only Timestamps for consistent lookup
    return set(pd.Timestamp(d).normalize() for d in event_dates)

target_start = datetime(2015, 1, 1)
# Extend to cover the full original data range; use 2026-12-31 if original goes beyond 2025
original_max_date = df_original['Date'].max()
target_end = max(original_max_date, pd.Timestamp('2024-12-31')).to_pydatetime()

event_days = create_event_days(target_start, target_end, n_events=100, seed=42)

print(f"Synthetic period: {target_start.date()} to {target_end.date()}")
print(f"Selected {len(event_days)} event days")
print("First 5:", sorted(event_days)[:5])

Synthetic period: 2015-01-01 to 2025-08-10
Selected 100 event days
First 5: [Timestamp('2015-01-15 00:00:00'), Timestamp('2015-02-02 00:00:00'), Timestamp('2015-07-04 00:00:00'), Timestamp('2015-07-16 00:00:00'), Timestamp('2015-08-09 00:00:00')]


## 7. Generate Synthetic Records
We loop over each day and for each meal generate a batch of events. Each event is assigned a 30-minute-aligned timestamp, categorical values, waste weight drawn from the calibrated log-normal distribution, foot traffic value, and derived features.

After generating event-level rows we also **fill every 30-minute bin** in the meal windows with a minimum background reading (small positive noise drawn from the lower percentile of the original distribution) so there are no completely empty intervals in the final time series.

In [87]:
# Background fill uses the SAME log-normal params as event records.
# Using mean-std was pulling the overall median down from 2.6 to 1.05.
# Background slots represent real (lower-activity) waste, not near-zero noise.
bg_params = {}
for cat in df_original["Food_Category"].unique():
    log_vals = df_original[df_original["Food_Category"] == cat]["Log_Waste"]
    bg_params[cat] = {
        "mean_log": log_vals.mean(),   # same centre as event records
        "std_log": log_vals.std()       # same spread
    }

def generate_background_waste(category):
    """Draw a waste value for an uncovered 30-min bin from the same log-normal
    as event records. Capped at 5 kg to stay in-distribution."""
    p = bg_params[category]
    waste = np.exp(np.random.normal(p["mean_log"], p["std_log"]))
    return max(min(waste, 5.0), 0.01)

# All 30-min slots within each meal window
def meal_30min_slots(meal):
    """Return list of (hour, minute) pairs covering the meal window in 30-min steps."""
    start, end = meal_time_windows[meal]
    slots = []
    h = start
    while h < end:
        slots.append((h, 0))
        slots.append((h, 30))
        h += 1
    return slots

print("30-min slots per meal:")
for m in meal_time_windows:
    print(f"  {m}: {meal_30min_slots(m)}")


30-min slots per meal:
  Breakfast: [(6, 0), (6, 30), (7, 0), (7, 30), (8, 0), (8, 30)]
  Lunch: [(11, 0), (11, 30), (12, 0), (12, 30), (13, 0), (13, 30)]
  Dinner: [(17, 0), (17, 30), (18, 0), (18, 30), (19, 0), (19, 30)]


In [88]:
synthetic_rows = []
date_range_gen = pd.date_range(target_start, target_end, freq='D')

for date in date_range_gen:
    date_norm = date.normalize()  # date-only for event_days lookup
    is_event = date_norm in event_days

    # Foot traffic multiplier (for foot_traffic column and event count scaling)
    day_multiplier = get_foot_traffic_multiplier(date_norm, event_days)

    for meal in meal_dist.index:
        # Track which 30-min slots already have an event record
        covered_slots = set()

        # ---- Event-based records ----------------------------------------
        n_events = generate_event_count(meal, date.weekday(), day_multiplier, baseline_events)

        for _ in range(n_events):
            section  = np.random.choice(section_dist.index, p=section_dist.values)
            cat_probs = category_given_meal.loc[meal]
            category  = np.random.choice(cat_probs.index, p=cat_probs.values)

            timestamp = generate_event_timestamp(date, meal)
            slot_key  = (timestamp.hour, timestamp.minute)
            covered_slots.add(slot_key)

            hour_mult    = get_hour_multiplier(timestamp.hour)
            foot_traffic = base_traffic_per_section[section] * hour_mult * day_multiplier

            # Generate waste weight calibrated to original distribution
            waste_weight = generate_waste_weight(category, is_event_day=is_event)
            unit_price   = price_map[category]

            row = {
                'Date': timestamp,
                'Meal': meal,
                'Canteen_Section': section,
                'Food_Category': category,
                'Waste_Weight_kg': round(waste_weight, 2),
                'Unit_Price_per_kg': unit_price,
                'Is_Event_Day': int(is_event)
            }
            row.update(generate_derived_features(timestamp, waste_weight, unit_price, foot_traffic))
            synthetic_rows.append(row)

        # ---- Background fill: one record per uncovered 30-min slot ------
        for (h, m) in meal_30min_slots(meal):
            if (h, m) in covered_slots:
                continue  # already has an event record

            timestamp = date + timedelta(hours=h, minutes=m)
            section   = np.random.choice(section_dist.index, p=section_dist.values)
            cat_probs = category_given_meal.loc[meal]
            category  = np.random.choice(cat_probs.index, p=cat_probs.values)

            hour_mult    = get_hour_multiplier(h)
            foot_traffic = base_traffic_per_section[section] * hour_mult * day_multiplier

            waste_weight = generate_background_waste(category)
            unit_price   = price_map[category]

            row = {
                'Date': timestamp,
                'Meal': meal,
                'Canteen_Section': section,
                'Food_Category': category,
                'Waste_Weight_kg': round(waste_weight, 2),
                'Unit_Price_per_kg': unit_price,
                'Is_Event_Day': int(is_event)
            }
            row.update(generate_derived_features(timestamp, waste_weight, unit_price, foot_traffic))
            synthetic_rows.append(row)

df_synthetic = pd.DataFrame(synthetic_rows)
print(f"Synthetic data shape: {df_synthetic.shape}")
print("\nSynthetic Waste_Weight_kg stats:")
print(df_synthetic['Waste_Weight_kg'].describe())

Synthetic data shape: (76290, 23)

Synthetic Waste_Weight_kg stats:
count    76290.000000
mean         2.424110
std          1.489118
min          0.040000
25%          1.190000
50%          2.050000
75%          3.540000
max          5.000000
Name: Waste_Weight_kg, dtype: float64


## 7b. Validate Scale Alignment
Before combining, we verify that the synthetic waste distribution closely matches the original. The mean, median, and standard deviation should be in the same range.

In [89]:
print("=== Scale validation ===")
print(f"{'Metric':<15} {'Original':>12} {'Synthetic':>12}")
print("-" * 40)
for stat in ['mean', 'median', 'std', 'min', 'max']:
    orig_val = getattr(df_original['Waste_Weight_kg'], stat)()
    synt_val = getattr(df_synthetic['Waste_Weight_kg'], stat)()
    flag = "  ← CHECK" if abs(orig_val - synt_val) / max(orig_val, 1e-9) > 0.5 else ""
    print(f"{stat:<15} {orig_val:>12.3f} {synt_val:>12.3f}{flag}")

=== Scale validation ===
Metric              Original    Synthetic
----------------------------------------
mean                   2.586        2.424
median                 2.600        2.050
std                    1.408        1.489
min                    0.100        0.040  ← CHECK
max                    5.000        5.000


## 8. Add Flag Columns to Synthetic Data
We apply the same outlier and high-waste thresholds to the synthetic records to maintain consistency with the original data.

In [90]:
df_synthetic = apply_flag_thresholds(df_synthetic, thresholds)
print("Flag columns added.")

Flag columns added.


## 9. Combine Original and Synthetic Data
We merge the original dataset (with added realistic times) and the synthetic dataset. Both now have the same structure and flags. We compute foot traffic for the original rows using the same logic, add the `Is_Event_Day` flag, and sort chronologically.

In [91]:
# df_original_clean = df_original.copy()

# # Columns to add to original if missing
# if 'Hour' not in df_original_clean.columns:
#     df_original_clean['Hour']         = df_original_clean['Date'].dt.hour
# if 'Minute' not in df_original_clean.columns:
#     df_original_clean['Minute']       = df_original_clean['Date'].dt.minute
# if 'DayOfYear' not in df_original_clean.columns:
#     df_original_clean['DayOfYear']    = df_original_clean['Date'].dt.dayofyear
# if 'Weekday' not in df_original_clean.columns:
#     df_original_clean['Weekday']      = df_original_clean['Date'].dt.day_name()
# if 'Week' not in df_original_clean.columns:
#     df_original_clean['Week']         = df_original_clean['Date'].dt.isocalendar().week
# if 'Quarter' not in df_original_clean.columns:
#     df_original_clean['Quarter']      = df_original_clean['Date'].dt.quarter
# if 'IsWeekend' not in df_original_clean.columns:
#     df_original_clean['IsWeekend']    = (df_original_clean['Date'].dt.dayofweek >= 5).astype(int)
# if 'Month_Name' not in df_original_clean.columns:
#     df_original_clean['Month_Name']   = df_original_clean['Date'].dt.strftime('%B')
# if 'Weekday_Type' not in df_original_clean.columns:
#     df_original_clean['Weekday_Type'] = df_original_clean['IsWeekend'].map({0: 'Weekday', 1: 'Weekend'})
# if 'Waste_per_Price' not in df_original_clean.columns:
#     df_original_clean['Waste_per_Price'] = df_original_clean['Waste_Weight_kg'] / df_original_clean['Unit_Price_per_kg']
# if 'Log_Waste' not in df_original_clean.columns:
#     df_original_clean['Log_Waste']    = np.log(df_original_clean['Waste_Weight_kg'])

# # Foot traffic for original rows
# def compute_foot_traffic(row):
#     section   = row['Canteen_Section']
#     hour      = row['Date'].hour
#     date_norm = row['Date'].normalize()
#     day_mult  = get_foot_traffic_multiplier(date_norm, event_days)
#     hour_mult = get_hour_multiplier(hour)
#     return base_traffic_per_section[section] * hour_mult * day_mult

# if 'Foot_Traffic' not in df_original_clean.columns:
#     df_original_clean['Foot_Traffic'] = df_original_clean.apply(compute_foot_traffic, axis=1)

# # Is_Event_Day flag for original rows
# df_original_clean['Is_Event_Day'] = df_original_clean['Date'].dt.normalize().isin(event_days).astype(int)

# # Re-apply flag thresholds to original
# df_original_clean = apply_flag_thresholds(df_original_clean, thresholds)

# # Align columns: keep only the union; fill any truly missing columns with NaN
# all_cols = list(dict.fromkeys(list(df_original_clean.columns) + list(df_synthetic.columns)))
# for col in all_cols:
#     if col not in df_original_clean.columns:
#         df_original_clean[col] = np.nan
#     if col not in df_synthetic.columns:
#         df_synthetic[col] = np.nan

# Combine and sort
# df_synthetic = pd.concat([df_original_clean[all_cols], df_synthetic[all_cols]], ignore_index=True)
# df_synthetic = df_synthetic.sort_values('Date').reset_index(drop=True)

# print(f"Combined data shape: {df_synthetic.shape}")
# print("\nFoot traffic summary:")
# print(df_synthetic['Foot_Traffic'].describe())
# print("\nCombined Waste_Weight_kg stats:")
# print(df_synthetic['Waste_Weight_kg'].describe())

## 10. Save Augmented Dataset
Finally, we save the combined dataset to a CSV file. This file contains both original and synthetic records with full timestamps, foot traffic column, and all flags, ready for further analysis and modeling.

In [92]:
output_filename = 'data/food_waste_augmented_cl.csv'
df_synthetic.to_csv(output_filename, index=False)
print(f"File saved as {output_filename}")

File saved as data/food_waste_augmented_cl.csv


## 11. Quick Verification of Temporal Coverage
We check that:
1. The combined dataset covers the full target period.
2. The 30-minute grid has a high proportion of non-zero windows (target: >50% due to background fill).
3. The waste distribution in the synthetic data closely matches the original.

In [93]:
# Ensure datetime index
df_synthetic = df_synthetic.sort_values("Date")
agg = (
    df_synthetic.set_index("Date")
    .resample("30min")["Waste_Weight_kg"]
    .sum()
)

# Restrict zero-window check to meal hours only (non-meal hours are
# intentionally empty — the canteen is closed outside meal windows).
meal_hours = set()
for start, end in meal_time_windows.values():
    for h in range(start, end):
        meal_hours.add(h)

agg_meal = agg[agg.index.hour.isin(meal_hours)]

total_windows   = len(agg_meal)
zero_windows    = (agg_meal == 0).sum()
nonzero_windows = (agg_meal > 0).sum()

print(f"Total 30-min meal-hour windows: {total_windows}")
print(f"Zero windows (meal hours only): {zero_windows} ({zero_windows/total_windows*100:.1f}%)")
print(f"Non-zero windows:               {nonzero_windows} ({nonzero_windows/total_windows*100:.1f}%) ")
print(f"""
Note: non-meal hours (outside breakfast/lunch/dinner windows) are""")
print(f"""      intentionally zero — the canteen is closed during those times.""")
print(f"""
Date range in combined data: {df_synthetic['Date'].min()} to {df_synthetic['Date'].max()}""")

print("""
=== Final scale check ===""")
orig_mean   = df_original["Waste_Weight_kg"].mean()
synt_mean   = df_synthetic["Waste_Weight_kg"].mean()
comb_mean   = df_synthetic["Waste_Weight_kg"].mean()
print(f"Original  mean:   {orig_mean:.3f} kg  |  median: {df_original['Waste_Weight_kg'].median():.3f} kg  |  std: {df_original['Waste_Weight_kg'].std():.3f} kg  |  max: {df_original['Waste_Weight_kg'].max():.2f} kg")
print(f"Synthetic mean:   {synt_mean:.3f} kg  |  median: {df_synthetic['Waste_Weight_kg'].median():.3f} kg  |  std: {df_synthetic['Waste_Weight_kg'].std():.3f} kg  |  max: {df_synthetic['Waste_Weight_kg'].max():.2f} kg")
print(f"Combined  mean:   {comb_mean:.3f} kg")


Total 30-min meal-hour windows: 69750
Zero windows (meal hours only): 0 (0.0%)
Non-zero windows:               69750 (100.0%) 

Note: non-meal hours (outside breakfast/lunch/dinner windows) are
      intentionally zero — the canteen is closed during those times.

Date range in combined data: 2015-01-01 06:00:00 to 2025-08-10 19:30:00

=== Final scale check ===
Original  mean:   2.586 kg  |  median: 2.600 kg  |  std: 1.408 kg  |  max: 5.00 kg
Synthetic mean:   2.424 kg  |  median: 2.050 kg  |  std: 1.489 kg  |  max: 5.00 kg
Combined  mean:   2.424 kg
